# ViLegal_QRHE: Đánh giá mô hình trên tập Test
 

In [ ]:
%pip install -U  transformers datasets peft accelerate huggingface_hub safetensors  torch torchvision torchaudio torchao

In [ ]:
# %pip install -U accelerate 
%pip install -U datasets


# Mount drive

In [ ]:
IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    



# Import lib

In [ ]:
import os
import json, torch
import pandas as pd
import json
import os
from pathlib import Path
from dotenv import load_dotenv

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model

 

In [ ]:


BASE_MODEL = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
OUTPUT_DIR = "./legal-lora"
HF_USERNAME = "tmtdev"
HF_REPO = f"{HF_USERNAME}/vn-legal-model"
HF_DATASET_REPO = f"{HF_USERNAME}/vn-legal-extraction"

MAX_LENGTH = 1024
DATA_DIR = "/content/drive/MyDrive/legal-ai/data"
DATA_PATH = os.path.join(DATA_DIR, "all_extract_citation_data.json")


# Directory Path

# Config class

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM
import accelerate

print(accelerate.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"

# MODEL_ID = "ntphuc149/ViLegalQwen2.5-1.5B-Base"
HF_USERNAME = "tmtdev"
MODEL_ID = f"{HF_USERNAME}/vn-legal-model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

SYSTEM_PROMPT = """
Bạn là hệ thống trích xuất cấu trúc truy vấn pháp luật Việt Nam.

Đầu vào: query_text của người dùng.
Đầu ra: duy nhất một JSON hợp lệ theo cấu trúc:

{
  "query_text": "string",
  "question_text": "string",
  "target": {
    "part": null,
    "chapter": null,
    "section": null,
    "subsection": null,
    "article": null,
    "clause": null,
    "point": null,
    "document": null
  }
}

Quy tắc:
- query_text: giữ nguyên chính xác câu đầu vào.
- question_text: giữ nội dung câu hỏi, loại bỏ phần trích dẫn pháp luật
  như tên văn bản, Phần, Chương, Mục, Tiểu mục, Điều, Khoản, Điểm.
  Không làm thay đổi ý nghĩa câu hỏi.
- target: trích xuất chính xác các cấp trích dẫn xuất hiện trong query_text.
- document: tên văn bản pháp luật được nhắc đến trong query_text.
- Các cấp không được nhắc đến phải là null.
- Giá trị cấp trích dẫn là chuỗi, ví dụ "I", "13", "2", "a".
- Không suy đoán cấp trích dẫn còn thiếu.
- Không tự tra cứu hoặc tự bổ sung thông tin pháp luật.
- Chỉ trả về JSON, không thêm Markdown hay giải thích.
"""


SYSTEM_PROMPT = """
Bạn là trợ lý xử lý truy vấn pháp luật Việt Nam.
Dựa vào dữ liệu đã học, hãy tạo JSON hợp lệ gồm:
query_text, question_text, context.
Không thêm nội dung ngoài JSON.
"""

def extract_legal_structure(query_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query_text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        # return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Use inputs["input_ids"] to slice the output tokens correctly
    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("RAW OUTPUT:")
    print(result)

    try:
        parsed = json.loads(result)
        print("\nPARSED JSON:")
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
        return parsed
    except json.JSONDecodeError:
        print("\nModel chưa trả về JSON hợp lệ.")
        return None

extract_legal_structure(
    "Chương I của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam "
    "năm 2013 quy định về vấn đề gì?"
)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Tổng số tham số: {total_params:,}") 
# Kết quả sẽ ra khoảng ~1,540,000,000 (tức là 1.54B)

# Hoặc in thông tin chi tiết cấu hình mô hình
print(model.config)

In [ ]:
from datasets import load_dataset

raw_test = load_dataset(HF_DATASET_REPO, split="test")

print("Số lượng mẫu test:", len(raw_test))
print("Ví dụ mẫu test:", raw_test[0])